In [1]:
import importlib
import pandas as pd
import numpy as np
import re
from pathlib import Path

import functions
import functions_GLM
importlib.reload(functions)
importlib.reload(functions_GLM)
from functions import *
from functions_GLM import *

df_1111 = pd.read_csv("AI_Lit_Que_1111.csv")
df_1204 = pd.read_csv("AI_Lit_Que_1204.csv")

# add source/course label
df_1111["course"] = "1111"
df_1204["course"] = "1204"

print(df_1111.shape, df_1204.shape)

df, meta = prepare_dataset("AI_Lit_Que_1111.csv", "AI_Lit_Que_1204.csv")


(118, 37) (23, 37)


# Generalized linear model
$$
M_i = \beta_0 + \beta_1 A_i + \boldsymbol{\beta}_c^\top C_i + \varepsilon_i
$$

$$
\log\{P(Y_i=1 \mid A_i, M_i, C_i)\}
= \theta_0 + \theta_1 A_i + \theta_2 M_i + \boldsymbol{\theta}_c^\top C_i
$$

where $Y_i$ is the binary AI literacy(High or low) variable, $A_i$ is the SES variable, $M_i$ is the mediator variable.


In [2]:
# add mediator
df_analysis = add_mediator_composites(df, mediator_map)

#efa
ai_efa_items = [
    "ai_concept_data_bias_scored_num",
    "ai_concept_blackbox_scored_num",
    "ai_concept_input_variation_scored_num",
    "ai_concept_prompt_wording_scored_num",
    "ai_concept_social_ethics_scored_num",
    "ai_ability_training_data_scored_num",
    "ai_ability_explainability_scored_num",
    "ai_ability_input_sensitivity_scored_num",
    "ai_ability_prompting_scored_num",
    "ai_ability_social_ethics_scored_num",
]

fa_combined, d_combined, loadings_combined, variance_combined = fit_efa(
    df=df,
    sample="Combined",
    items=ai_efa_items,
    n_factors=2,
    rotation="oblimin"
)

scores = fa_combined.transform(d_combined)
df_factors = pd.DataFrame(
    scores,
    index=d_combined.index,
    columns=["ai_factor1_score", "ai_factor2_score"]
)

if "ai_factor1_score" in df_factors.columns and "ai_factor2_score" in df_factors.columns:
    df_analysis.loc[df_factors.index, "ai_factor1_score"] = df_factors["ai_factor1_score"]
    df_analysis.loc[df_factors.index, "ai_factor2_score"] = df_factors["ai_factor2_score"]

df_analysis = add_ai_course_background(df_analysis)
df_analysis["ai_course_yes"].value_counts(dropna=False)

df_bin, cut1 = make_binary_outcome(
    df_analysis,
    source_col="ai_factor1_score",
    new_col="ai_factor1_high",
    threshold="median",
    higher_is_one=True
)


In [19]:
mediators = [
    "language_load_score",
    "epistemic_stance_score",
    "practical_ai_use_score",
    "learning_ecology_score",
    "conceptual_exposure_score"
]

poisson_results = run_poisson_mediations(
    df=df_bin,
    x="ses_index",
    mediators=mediators,
    y_bin="ai_factor1_high",
    sample="Combined",
    covariates=[],
    n_boot=2000,
    seed=2026
)

poisson_results.round(3)

,sample,mediator,outcome,n,a_path,a_p,theta2_mediator_logrr,theta2_p,direct_logrr,direct_rr,...,indirect_logrr,indirect_rr,r2_mediator_model,indirect_logrr_boot_mean,indirect_logrr_ci_low_95,indirect_logrr_ci_high_95,indirect_rr_boot_mean,indirect_rr_ci_low_95,indirect_rr_ci_high_95,prop_logrr_positive
0,Combined,language_load_score,ai_factor1_high,141,-0.159,0.068,-0.190,0.040,0.070,1.073,...,0.030,1.031,0.025,0.031,-0.004,0.085,1.031,0.996,1.089,0.956
1,Combined,epistemic_stance_score,ai_factor1_high,141,0.111,0.162,0.194,0.012,0.076,1.079,...,0.021,1.022,0.012,0.021,-0.008,0.064,1.021,0.992,1.066,0.910
2,Combined,practical_ai_use_score,ai_factor1_high,141,0.109,0.218,0.023,0.794,0.098,1.103,...,0.002,1.002,0.012,0.001,-0.026,0.029,1.001,0.975,1.029,0.541
3,Combined,learning_ecology_score,ai_factor1_high,141,0.239,0.002,0.039,0.650,0.091,1.096,...,0.009,1.009,0.057,0.010,-0.036,0.058,1.010,0.964,1.059,0.681
4,Combined,conceptual_exposure_score,ai_factor1_high,141,0.219,0.006,-0.098,0.240,0.123,1.131,...,-0.022,0.979,0.048,-0.020,-0.063,0.019,0.980,0.939,1.019,0.121


- It gave similar story as continuous outcome model.

-----
## With interaction effect

$$
\log(RR_{IE}(a)) = \beta_1 \left(\theta_2 + \theta_3 a\right)
$$

$$
RR_{IE}(a) = \exp\left[\beta_1 \left(\theta_2 + \theta_3 a\right)\right]
$$


In [6]:
df_bin, cut1 = make_binary_outcome(
    df_analysis,
    source_col="ai_factor2_score",
    new_col="ai_factor2_high",
    threshold="median",
    higher_is_one=True
)

mediators = [
    "language_load_score",
    "epistemic_stance_score",
    "practical_ai_use_score",
    "learning_ecology_score",
    "conceptual_exposure_score",
]

poisson_interaction_results = run_poisson_mediations_interaction(
    df=df_bin,
    x="ses_index",
    mediators=mediators,
    y_bin="ai_factor2_high",
    sample="Combined",
    covariates=[],
    eval_at_x=0.0,   # indirect effect at average SES
    n_boot=2000,
    seed=2026
)

poisson_interaction_results.round(3)



,sample,mediator,outcome,n,a_path,a_p,theta1_direct_logrr,theta1_direct_p,theta2_mediator_logrr,theta2_p,...,indirect_rr_eval,eval_at_x,r2_mediator_model,indirect_logrr_boot_mean,indirect_logrr_ci_low_95,indirect_logrr_ci_high_95,indirect_rr_boot_mean,indirect_rr_ci_low_95,indirect_rr_ci_high_95,prop_logrr_positive
0,Combined,language_load_score,ai_factor2_high,141,-0.159,0.068,0.081,0.387,-0.205,0.026,...,1.033,0.0,0.025,0.034,-0.001,0.090,1.035,0.999,1.094,0.969
1,Combined,epistemic_stance_score,ai_factor2_high,141,0.111,0.162,0.106,0.234,0.208,0.019,...,1.023,0.0,0.012,0.023,-0.008,0.074,1.023,0.992,1.076,0.908
2,Combined,practical_ai_use_score,ai_factor2_high,141,0.109,0.218,0.117,0.190,0.098,0.243,...,1.011,0.0,0.012,0.010,-0.013,0.043,1.010,0.987,1.044,0.791
3,Combined,learning_ecology_score,ai_factor2_high,141,0.239,0.002,0.109,0.231,0.105,0.212,...,1.026,0.0,0.057,0.025,-0.017,0.073,1.025,0.983,1.076,0.898
4,Combined,conceptual_exposure_score,ai_factor2_high,141,0.219,0.006,0.152,0.099,-0.055,0.530,...,0.988,0.0,0.048,-0.013,-0.058,0.026,0.987,0.944,1.026,0.228


- nothing new

- The direct and total effects coefficients are not significant.